# Sinusoidal Position Encoding

**难度:** Easy

实现《Attention Is All You Need》中的正弦位置编码。

位置编码加到词嵌入上，让模型感知 token 的位置信息。原始 Transformer 使用固定的正弦函数。

**签名:** `sinusoidal_pe(seq_len, d_model) -> Tensor`

**参数:**
- `seq_len` — 位置数量
- `d_model` — 嵌入维度（必须为偶数）

**返回:** 形状为 `(seq_len, d_model)` 的位置编码张量

**公式:**
- `PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))`
- `PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))`

**提示:**
1. `freqs = 1 / 10000^(2i/d_model)`，i 取 `range(d_model//2)`
2. `angles = pos[:, None] * freqs[None, :]` → 形状 `(seq_len, d_model//2)`
3. `pe[:, 0::2] = sin(angles)`，`pe[:, 1::2] = cos(angles)`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写正弦位置编码 ----
# 面试要点：理解公式含义 + 广播机制

def sinusoidal_pe(seq_len, d_model):
    # 位置：[0, 1, ..., seq_len-1]
    pos = torch.arange(seq_len).float().unsqueeze(1)  # (seq_len, 1)

    # 维度索引：[0, 2, 4, ..., d_model-2]
    dim = torch.arange(0, d_model, 2).float()  # (d_model//2,)

    # 面试考点：为什么用 10000^(2i/d) 做分母？
    # 答：2i/d_model 从 0 到 ~1，所以频率从 1 到 1/10000
    # 这形成了一个"频率谱"：低维度变化快（编码局部位置），高维度变化慢（编码全局位置）
    # 类似傅里叶变换用不同频率的基函数表示信号
    freqs = 1.0 / (10000.0 ** (dim / d_model))

    # 广播：(seq_len, 1) * (d_model//2,) → (seq_len, d_model//2)
    angles = pos * freqs

    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(angles)  # 偶数维 sin
    pe[:, 1::2] = torch.cos(angles)  # 奇数维 cos

    # 面试考点：为什么 sin 和 cos 配对？
    # 答：sin 和 cos 是正交的，可以区分位置
    # 数学上，[sin(θ), cos(θ)] 可以看作单位圆上的点
    # 两个位置的内积只依赖于相对距离，这就是 RoPE 的前身
    return pe

In [ ]:
from torch_judge import check
check("sinusoidal_pe")

## 📝 核心思路总结

1. **正弦编码用不同频率的 sin/cos 表示位置**：低维度高频（局部区分），高维度低频（全局位置）
2. **偶数维用 sin，奇数维用 cos**：同一个频率的 sin 和 cos 配对，形成旋转编码
3. **无需学习，可外推到更长序列**：相比学习式位置编码，正弦编码天然支持任意长度